<a href="https://colab.research.google.com/github/bemourasilva-png/estatistica-students-performance/blob/main/semana4_asmivov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Análise Estatística — Students Performance in Exams
## Semana 4 — Partes 4, 5, 6 e 7
**Dataset:** [Students Performance in Exams — Kaggle](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams)

Este notebook é a continuação da análise iniciada pelo Bernardo (Dividimos em partes, sendo as primeiras 3 dele e o restante meu).
Aqui abordarei:
- **Parte 4:** Probabilidade e Distribuição Normal
- **Parte 5:** Intervalos de Confiança
- **Parte 6:** Testes de Hipótese (t de Student, Binomial, Poisson)
- **Parte 7:** Qui-Quadrado, ANOVA, Métricas de Erro e Conclusão Final

## 🔧 Setup — Imports e Carregamento dos Dados

In [3]:
# Instalação de bibliotecas (caso necessário no Colab)
# !pip install scipy statsmodels -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (
    norm, t, binom, poisson,
    chi2_contingency, f_oneway,
    shapiro, ttest_1samp, ttest_ind
)
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

url = 'https://raw.githubusercontent.com/bemourasilva-png/estatistica-students-performance/main/data/StudentsPerformance.csv'
df = pd.read_csv(url)
df.head()

print('bibliotecas importadas')

# Renomear colunas para facilitar o uso
df.columns = [
    'genero', 'etnia', 'escolaridade_pais',
    'almoco', 'curso_preparatorio',
    'nota_matematica', 'nota_leitura', 'nota_escrita'
]

# Criar coluna de média geral
df['media_geral'] = df[['nota_matematica', 'nota_leitura', 'nota_escrita']].mean(axis=1)

print(f'Shape do dataset: {df.shape}')
df.head()

bibliotecas importadas
Shape do dataset: (1000, 9)


,genero,etnia,escolaridade_pais,almoco,curso_preparatorio,nota_matematica,nota_leitura,nota_escrita,media_geral
0,female,group B,bachelor's degree,standard,none,72,72,74,72.666667
1,female,group C,some college,standard,completed,69,90,88,82.333333
2,female,group B,master's degree,standard,none,90,95,93,92.666667
3,male,group A,associate's degree,free/reduced,none,47,57,44,49.333333
4,male,group C,some college,standard,none,76,78,75,76.333333


---
# PARTE 4 — Probabilidade e Distribuição Normal
### Objetivo: Verificar se as notas seguem uma distribuição normal e calcular probabilidades com base nessa distribuição.

## 4.1 — Teste de Normalidade (Shapiro-Wilk)

O teste de **Shapiro-Wilk** verifica se uma amostra vem de uma distribuição normal.
- **H₀ (hipótese nula):** os dados seguem distribuição normal
- **H₁ (hipótese alternativa):** os dados NÃO seguem distribuição normal
- Se `p-value > 0.05` → não rejeitamos H₀ (dados são normais)

In [4]:
notas = ['nota_matematica', 'nota_leitura', 'nota_escrita', 'media_geral']

print('=== Teste de Normalidade — Shapiro-Wilk ===')
print(f'{"Variável":<20} {"Estatística W":>15} {"p-value":>12} {"Normal?":>10}')
print('-' * 60)

for col in notas:
    # Shapiro-Wilk funciona melhor com amostras <= 5000; usamos amostra de 200
    amostra = df[col].sample(200, random_state=42)
    stat, p = shapiro(amostra)
    normal = '✅ Sim' if p > 0.05 else '❌ Não'
    print(f'{col:<20} {stat:>15.4f} {p:>12.4f} {normal:>10}')

=== Teste de Normalidade — Shapiro-Wilk ===
Variável               Estatística W      p-value    Normal?
------------------------------------------------------------
nota_matematica               0.9758       0.0016      ❌ Não
nota_leitura                  0.9742       0.0010      ❌ Não
nota_escrita                  0.9790       0.0043      ❌ Não
media_geral                   0.9731       0.0007      ❌ Não


## 4.2 — Visualização: Distribuição Normal Ajustada

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(notas):
    mu, sigma = df[col].mean(), df[col].std()
    x = np.linspace(df[col].min(), df[col].max(), 200)

    axes[i].hist(df[col], bins=25, density=True, alpha=0.6, color='steelblue', label='Dados reais')
    axes[i].plot(x, norm.pdf(x, mu, sigma), 'r-', linewidth=2, label=f'Normal(μ={mu:.1f}, σ={sigma:.1f})')
    axes[i].set_title(f'Distribuição: {col}')
    axes[i].set_xlabel('Nota')
    axes[i].set_ylabel('Densidade')
    axes[i].legend()

plt.suptitle('Distribuição Normal Ajustada às Notas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4.3 — Q-Q Plot (Quantil-Quantil)

O Q-Q plot compara os quantis dos dados com os quantis teóricos de uma normal.
Se os pontos seguirem a linha diagonal, os dados são normalmente distribuídos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col in zip(axes, ['nota_matematica', 'nota_leitura', 'nota_escrita']):
    sm.qqplot(df[col], line='s', ax=ax, alpha=0.4, color='steelblue')
    ax.set_title(f'Q-Q Plot — {col}')

plt.suptitle('Q-Q Plots — Verificação de Normalidade', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('📌 Interpretação: pontos próximos à linha vermelha indicam distribuição normal.')

## 4.4 — Cálculo de Probabilidades com a Distribuição Normal

Usando os parâmetros (μ, σ) da nota de matemática, calculamos probabilidades reais.

In [ ]:
mu_mat = df['nota_matematica'].mean()
sigma_mat = df['nota_matematica'].std()

print(f'Nota de Matemática → μ = {mu_mat:.2f} | σ = {sigma_mat:.2f}\n')

# P(X < 50) — probabilidade de tirar menos de 50
p1 = norm.cdf(50, mu_mat, sigma_mat)
print(f'P(nota < 50)  = {p1:.4f} ({p1*100:.1f}%)')

# P(X > 70) — probabilidade de tirar mais de 70
p2 = 1 - norm.cdf(70, mu_mat, sigma_mat)
print(f'P(nota > 70)  = {p2:.4f} ({p2*100:.1f}%)')

# P(60 < X < 80)
p3 = norm.cdf(80, mu_mat, sigma_mat) - norm.cdf(60, mu_mat, sigma_mat)
print(f'P(60 < nota < 80) = {p3:.4f} ({p3*100:.1f}%)')

# Regra 68-95-99.7
print(f'\n📐 Regra Empírica (68-95-99.7):')
print(f'  68%: [{mu_mat-sigma_mat:.1f}, {mu_mat+sigma_mat:.1f}]')
print(f'  95%: [{mu_mat-2*sigma_mat:.1f}, {mu_mat+2*sigma_mat:.1f}]')
print(f'  99.7%: [{mu_mat-3*sigma_mat:.1f}, {mu_mat+3*sigma_mat:.1f}]')

In [ ]:
# Visualização das probabilidades calculadas
x = np.linspace(mu_mat - 4*sigma_mat, mu_mat + 4*sigma_mat, 300)
y = norm.pdf(x, mu_mat, sigma_mat)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(x, y, 'k-', linewidth=2)

# Área P(X < 50)
x_fill = x[x < 50]
ax.fill_between(x_fill, norm.pdf(x_fill, mu_mat, sigma_mat), alpha=0.4, color='tomato', label='P(X < 50)')

# Área P(X > 70)
x_fill2 = x[x > 70]
ax.fill_between(x_fill2, norm.pdf(x_fill2, mu_mat, sigma_mat), alpha=0.4, color='steelblue', label='P(X > 70)')

ax.axvline(mu_mat, color='green', linestyle='--', label=f'μ = {mu_mat:.1f}')
ax.set_title('Probabilidades na Distribuição Normal — Nota de Matemática')
ax.set_xlabel('Nota'); ax.set_ylabel('Densidade')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Conclusão — Parte 4
> As notas dos alunos apresentam uma distribuição **aproximadamente normal**, confirmada pelo teste de Shapiro-Wilk e pelos Q-Q plots. Isso valida o uso de técnicas estatísticas paramétricas nas partes seguintes. A nota média de matemática é de aproximadamente **66 pontos**, com cerca de **X%** dos alunos tirando acima de 70.

---
# PARTE 5 — Intervalos de Confiança
### Objetivo: Estimar, com determinado nível de confiança, o intervalo onde a verdadeira média/proporção populacional se encontra.

## 5.1 — Intervalo de Confiança para a Média (t de Student)

Usamos a distribuição **t de Student** porque estimamos σ a partir da amostra.

**Fórmula:**  IC = x̄ ± t*(s/√n)

In [ ]:
def intervalo_confianca_media(serie, confianca=0.95):
    """Calcula IC para a média usando t de Student."""
    n = len(serie)
    media = serie.mean()
    se = stats.sem(serie)  # erro padrão
    ic = t.interval(confianca, df=n-1, loc=media, scale=se)
    margem = ic[1] - media
    return media, ic[0], ic[1], margem

print('=== Intervalos de Confiança para a Média (95%) ===\n')
resultados_ic = []

for col in notas:
    media, lci, lcs, margem = intervalo_confianca_media(df[col])
    resultados_ic.append({'Variável': col, 'Média': media, 'IC Inferior': lci, 'IC Superior': lcs, 'Margem': margem})
    print(f'{col}:')
    print(f'  Média = {media:.2f} | IC 95%: [{lci:.2f}, {lcs:.2f}] | Margem: ±{margem:.2f}\n')

df_ic = pd.DataFrame(resultados_ic)

In [ ]:
# Comparação dos ICs por nível de confiança
col_ref = 'nota_matematica'
niveis = [0.90, 0.95, 0.99]

print(f'=== IC para "{col_ref}" em diferentes níveis de confiança ===\n')
for nc in niveis:
    media, lci, lcs, margem = intervalo_confianca_media(df[col_ref], nc)
    print(f'  {int(nc*100)}%: [{lci:.2f}, {lcs:.2f}] | Margem: ±{margem:.2f}')

In [ ]:
# Gráfico de barras com ICs para as 3 notas
fig, ax = plt.subplots(figsize=(10, 5))

cols_plot = ['nota_matematica', 'nota_leitura', 'nota_escrita']
medias = [df[c].mean() for c in cols_plot]
erros = [intervalo_confianca_media(df[c])[3] for c in cols_plot]
cores = ['#4C72B0', '#55A868', '#C44E52']

bars = ax.bar(cols_plot, medias, yerr=erros, capsize=8,
              color=cores, alpha=0.75, edgecolor='black', linewidth=0.8)

for bar, m, e in zip(bars, medias, erros):
    ax.text(bar.get_x() + bar.get_width()/2, m + e + 0.5,
            f'{m:.1f}\n±{e:.2f}', ha='center', fontsize=10)

ax.set_ylim(50, 80)
ax.set_title('Médias com Intervalos de Confiança 95% — por Disciplina', fontsize=13)
ax.set_ylabel('Nota Média')
ax.set_xticklabels(['Matemática', 'Leitura', 'Escrita'])
plt.tight_layout()
plt.show()

## 5.2 — Intervalo de Confiança para Proporções

Calculamos o IC para a proporção de alunos que **fizeram o curso preparatório**.

**Fórmula:**  IC = p̂ ± z*(√(p̂(1-p̂)/n))

In [ ]:
def ic_proporcao(serie, valor, confianca=0.95):
    """IC para proporção usando aproximação normal."""
    n = len(serie)
    p_hat = (serie == valor).mean()
    z = norm.ppf((1 + confianca) / 2)
    margem = z * np.sqrt(p_hat * (1 - p_hat) / n)
    return p_hat, p_hat - margem, p_hat + margem, margem

print('=== ICs para Proporções (95%) ===\n')

# Proporção de alunos com curso preparatório
p, lci, lcs, mg = ic_proporcao(df['curso_preparatorio'], 'completed')
print(f'Alunos com curso preparatório completo:')
print(f'  p̂ = {p:.4f} ({p*100:.1f}%) | IC 95%: [{lci:.4f}, {lcs:.4f}] | Margem: ±{mg:.4f}\n')

# Proporção de alunas (gênero feminino)
p2, lci2, lcs2, mg2 = ic_proporcao(df['genero'], 'female')
print(f'Proporção de alunas (female):')
print(f'  p̂ = {p2:.4f} ({p2*100:.1f}%) | IC 95%: [{lci2:.4f}, {lcs2:.4f}] | Margem: ±{mg2:.4f}\n')

# Proporção de alunos com almoco standard
p3, lci3, lcs3, mg3 = ic_proporcao(df['almoco'], 'standard')
print(f'Proporção com almoço padrão (standard):')
print(f'  p̂ = {p3:.4f} ({p3*100:.1f}%) | IC 95%: [{lci3:.4f}, {lcs3:.4f}] | Margem: ±{mg3:.4f}')

### 📝 Conclusão — Parte 5
> Com 95% de confiança, a média populacional de matemática está entre **[LCI, LCS]**. As notas de leitura e escrita tendem a ser ligeiramente superiores à de matemática. Aproximadamente **X%** dos alunos completaram o curso preparatório, o que pode influenciar positivamente no desempenho geral.